# 01 Dataset Exploration\n
Load finance datasets from HuggingFace and inspect quality.

In [ ]:
from datasets import load_dataset
import json

print("Loading FinQA dataset from HuggingFace...")
finqa = load_dataset('ibm/finqa')
print("\n✓ Dataset loaded successfully!")
print(f"\nDataset structure:")
print(finqa)

In [ ]:
print("\n" + "="*60)
print("Dataset Statistics")
print("="*60)

for split in finqa.keys():
    print(f"\n{split.upper()} set:")
    print(f"  Samples: {len(finqa[split])}")
    print(f"  Features: {list(finqa[split].column_names)}")

# Sample inspection
sample = finqa['train'][0]
print(f"\nSample fields and types:")
for k, v in sample.items():
    if isinstance(v, str):
        v_preview = v[:100] + "..." if len(v) > 100 else v
        print(f"  {k}: str (len={len(v)})")
        print(f"    Preview: {v_preview}")
    else:
        print(f"  {k}: {type(v).__name__}")

In [ ]:
import os
from pathlib import Path

# Save dataset locally for faster loading in subsequent runs
data_dir = Path("../data/finqa")
data_dir.mkdir(parents=True, exist_ok=True)

print("Saving dataset locally...")
for split in finqa.keys():
    save_path = data_dir / split
    if not (save_path / "dataset_info.json").exists():
        finqa[split].save_to_disk(str(save_path))
        print(f"  ✓ Saved {split} split to {save_path}")
    else:
        print(f"  • {split} split already exists")

print(f"\nDataset saved to: {data_dir}")
print("\nDataset is ready for Phase 2: Data Preparation")

## 4. Save Dataset Locally for Next Phases

In [ ]:
import pandas as pd
import numpy as np

print("Data Quality Checks:")
print("-" * 60)

for split in ['train', 'validation', 'test']:
    if split not in finqa:
        continue
    
    dataset = finqa[split]
    print(f"\n{split.upper()}:")
    
    # Check for missing values
    missing_counts = {}
    for field in dataset.column_names:
        missing = sum(1 for item in dataset if item[field] is None or item[field] == '')
        if missing > 0:
            missing_counts[field] = missing
    
    if missing_counts:
        print(f"  Missing values: {missing_counts}")
    else:
        print(f"  ✓ No missing values")
    
    # Length statistics
    qa_lengths = [len(item['qa'][0].split()) for item in dataset]
    print(f"  Question length (words): min={min(qa_lengths)}, max={max(qa_lengths)}, avg={np.mean(qa_lengths):.1f}")
    
    ans_lengths = [len(str(item['qa'][1]).split()) for item in dataset]
    print(f"  Answer length (words): min={min(ans_lengths)}, max={max(ans_lengths)}, avg={np.mean(ans_lengths):.1f}")
    
    # Context length
    context_lengths = [len(item['pre_text'].split()) if item['pre_text'] else 0 for item in dataset]
    print(f"  Context length (words): min={min(context_lengths)}, max={max(context_lengths)}, avg={np.mean(context_lengths):.1f}")

## 3. Data Quality Analysis

In [ ]:
print("="*60)
print("Sample Training Data (First 2 examples)")
print("="*60)

for i in range(min(2, len(finqa['train']))):
    sample = finqa['train'][i]
    print(f"\nExample {i+1}:")
    print(f"Question: {sample['qa'][0]}")
    print(f"Answer: {sample['qa'][1]}")
    print(f"Financial Context (preview):")
    context_preview = sample['pre_text'][:300] if sample['pre_text'] else "N/A"
    print(f"  {context_preview}...")
    if 'table' in sample and sample['table']:
        print(f"Table data (preview): {str(sample['table'])[:200]}...")
    print()

## 2. Detailed Sample Inspection